# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=dwh_library;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2024;'
)

# Đọc data

## Đọc data từ SQL Server

In [3]:
# Hàm đọc dữ liệu từng phần và xử lý lỗi
def fetch_data_in_batches(query_base, connection, batch_size=100):
    offset = 0
    all_data = []  # Lưu tất cả các hàng hợp lệ
    while True:
        query = f"""
        {query_base}
        ORDER BY ID
        OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY
        """
        try:
            # Đọc dữ liệu batch hiện tại
            df_batch = pd.read_sql(query, connection)
            if df_batch.empty:  # Nếu không còn dữ liệu, dừng vòng lặp
                break
            all_data.append(df_batch)  # Lưu batch hợp lệ
            offset += batch_size  # Tăng offset để đọc batch tiếp theo
        except Exception as e:
            print(f"Lỗi xảy ra khi xử lý batch từ {offset}: {e}")
            offset += batch_size  # Bỏ qua batch bị lỗi và tiếp tục
    # Gộp tất cả các batch thành DataFrame duy nhất
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

In [ ]:
query_Xepgia = """
SELECT ID,
       Tai_lieu_ID, 
       Ma_xep_gia,
       Ten_thu_vien_ID,
       Kho_ID,
       Ngay_bo_sung,
       Cho_nhap_kho,
       InUsed,
       Gia,
       Gia_tien,
       InCirculation,
       Kiem_ke,
       dbo.DecodeUTF8String(Nguon_Nhap) AS Nguon_Nhap,
       dbo.DecodeUTF8String(Callnumber) AS Callnumber,
       So_HD
  FROM Ma_xep_gia
"""
df_xepgia = fetch_data_in_batches(query_Xepgia, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_xepgia)

C:\Users\phung\AppData\Local\Temp\ipykernel_11108\3076783788.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


### Tạo dataframe backup 

In [ ]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_xepgia.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_xepgia_backup = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_xepgia:", len(df_xepgia)) # Kiểm tra số lượng dòng
print("Số dòng trong df_xepgia_backup:", len(df_xepgia_backup)) # Kiểm tra số lượng dòng

Số dòng trong df_bandoc: 71421
Số dòng trong df_bandoc_backup: 71421


### [Nếu cần] lấy lại data từ backup

In [ ]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_xepgia_backup.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_xepgia = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_xepgia_backup:", len(df_xepgia_backup)) # Kiểm tra số lượng dòng
print("Số dòng trong df_xepgia:", len(df_xepgia)) # Kiểm tra số lượng dòng

Số dòng trong df_bandoc_backup: 71421
Số dòng trong df_bandoc: 71421


# Xử lý data

## Thêm 1 dòng giả định none

In [ ]:
# Tạo DataFrame `new_row` chứa dòng dữ liệu giả định
new_row = pd.DataFrame({
    'ID': [0],
    'Tai_lieu_ID': ['0'],
    'Ma_xep_gia': ['(Không xác định)'],
    'Ten_thu_vien_ID': ['0'],
    'Kho_ID': [0],
    'Ngay_bo_sung': ['1024-01-01 00:00:00'],
    'Cho_nhap_kho': [0],
    'InUsed': [0],
    'Gia': [0],
    'Gia_tien': [0],
    'InCirculation': [0],
    'Kiem_ke': [0],
    'Nguon_Nhap': ['(Không xác định)'],
    'Callnumber': ['(Không xác định)'],
    'So_HD': ['(Không xác định)']
})
# Thêm dòng dữ liệu giả định vào `df` bằng `pd.concat`
df_xepgia = pd.concat([df_xepgia, new_row], ignore_index=True) # Thêm vào dataframe
df_xepgia = df_xepgia.sort_values(by="ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_xepgia)

           So_the            Ho_ten            Ngay_sinh  Dan_toc_ID  \
0               0  (Không xác định)  1024-01-01 00:00:00          56   
1       0/8107029       Vũ Minh Đức  1989-05-02 00:00:00           1   
2        00000001    LƯU VĨNH QUANG  1976-01-01 00:00:00           1   
3        00000001    LƯU VĨNH QUANG  1976-01-01 00:00:00           1   
4        00000002     PHẠM HỮU TIẾN  1992-02-15 00:00:00           1   
...           ...               ...                  ...         ...   
71417   ÊU0401079       LÊ VĂN HIẾU                  NaT           1   
71418    Ô0105040        NG. TẤN ĐỘ                  NaT           1   
71419   ên2101158      ĐÀO MINH ĐỨC                  NaT           1   
71420   ƠI4401092      PHẠM VĂN LỢI  1980-05-31 00:00:00           1   
71421  ƠNG1113054  LÊ THỊ MỸ PHƯỢNG  1982-08-06 00:00:00           1   

       Trinh_do_ID So_dien_thoai Nghe_nghiep  \
0                0           NaN         NaN   
1                4                     

## Xử lý data rỗng hoặc " "

In [ ]:
df_xepgia = df_xepgia.replace('', None)
df_xepgia = df_xepgia.replace(np.nan, None)
print(df_xepgia)

      Nghe_nghiep                                 Co_quan chuc_vu
0            None                                    None    None
1            None                           Trường ĐHSPKT    None
2            None                           Trường ĐHSPKT    None
3            None  Trường TCN AN ĐỨC- TỔNG CONG TY LIKSIN    None
4            None                           Trường ĐHSPKT    None
...           ...                                     ...     ...
71417        None                                  ĐHSPKT    None
71418        None                                  ĐHSPKT    None
71419        None                                  ĐHSPKT    None
71420        None                                  ĐHSPKT    None
71421        None                                  ĐHSPKT    None

[71422 rows x 3 columns]


## Xử lý kiểu date

In [ ]:
query_date = "SELECT Date_key FROM DIM_Date"
df_date = pd.read_sql(query_date, conn_dwh_library)
date_ids = set(df_date['Date_key'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_xepgia['Ngay_bo_sung'] = pd.to_datetime(df_xepgia['Ngay_bo_sung'], errors='coerce')
df_xepgia['Ngay_bo_sung'] = df_xepgia['Ngay_bo_sung'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
print(df_xepgia[['Ngay_bo_sung']])

C:\Users\phung\AppData\Local\Temp\ipykernel_22068\4050017018.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_library)
C:\Users\phung\AppData\Local\Temp\ipykernel_22068\4050017018.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_bandoc['Ngay_sinh'] = pd.to_datetime(df_bandoc['Ngay_sinh'], errors='coerce')
C:\Users\phung\AppData\Local\Temp\ipykernel_22068\4050017018.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_bandoc['Ngay_cap'] = pd.to_datetime(df_bandoc['Ngay_cap'], errors='coerce')
C:\Users\phu

       Ngay_sinh  Ngay_cap  Ngay_het_han
0              0         0             0
1       19890502  20080926      20090831
2       19760101  20121227      20131227
3       19760101  20121227      20131227
4       19920215  20121228      20131228
...          ...       ...           ...
71417          0  20040106      20040831
71418          0  20020924      20030725
71419          0  20030826      20040831
71420   19800531  20040105      20040831
71421   19820806  20031129      20040831

[71422 rows x 3 columns]


## Xử lý ID_tai_lieu

In [ ]:
query_Tailieu = "SELECT ID_tai_lieu FROM DIM_Tai_lieu"
df_tailieu = pd.read_sql(query_Tailieu, conn_dwh_library)
tailieu_ids = set(df_tailieu['ID_tai_lieu'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong ID_tai_lieu của bảng DIM_Tai_lieu hay không ?
df_xepgia['Tai_lieu_ID'] = df_xepgia['Tai_lieu_ID'].apply(lambda x: x if pd.notna(x) and x in tailieu_ids else 0)
print(df_xepgia[['Tai_lieu_ID']])

## Xử lý ID_thu_vien

In [ ]:
# truy xuất dữ liệu từ libol ra
query_Thuvien = "SELECT Thu_vien_ID, dbo.DecodeUTF8String(Ten_viet_tat) AS Ten_viet_tat FROM Thu_vien"
df_thuvien = pd.read_sql(query_Thuvien, conn_libol)
map_dict = dict(zip(df_thuvien['Thu_vien_ID'], df_thuvien['Ten_viet_tat'])) # mapping lại 1:DHSPKT 2:ĐHSPKT
# kiểm tra id trong mapping nếu trùng với Thu_vien_ID thì đổi thành Ten_viet_tat
# vd 1 -> DHSPKT
df_xepgia['Ten_thu_vien_ID'] = df_xepgia['Ten_thu_vien_ID'].map(map_dict).fillna(df_xepgia['Ten_thu_vien_ID'])
# truy xuất dữ liệu từ dwh bảng DIM_Thu_vien ra
query_Thuvien = "SELECT ID_thu_vien FROM DIM_Thu_vien"
df_thuvien = pd.read_sql(query_Thuvien, conn_dwh_library)
thuvien_ids = set(df_thuvien['ID_thu_vien'])
# kiểm tra tên viết tắt ở trên kìa có trong bảng DIM_Thu_vien không? có thì bỏ qua không thì bằng 0
df_xepgia['Ten_thu_vien_ID'] = df_xepgia['Ten_thu_vien_ID'].apply(lambda x: x if x in thuvien_ids else 0)
print(df_xepgia[['Ten_thu_vien_ID']])

4             2010
5               00
6               00
7               00
8               00
           ...    
71416    2003-2005
71417    2000-2005
71419    2002-2007
71420    2000-2005
71421    2003-2008
Name: Khoa_hoc, Length: 69310, dtype: object


## xử lý ID_kho

In [ ]:
query_Kho = "SELECT ID_kho FROM DIM_Kho"
df_kho = pd.read_sql(query_Kho, conn_dwh_library)
kho_ids = set(df_kho['ID_kho'])
# chuyển date về dang int 
# kiểm tra id đó có tồn tại trong ID_kho của bảng DIM_Kho hay không ?
df_xepgia['Kho_ID'] = df_xepgia['Kho_ID'].apply(lambda x: x if pd.notna(x) and x in kho_ids else 0)
print(df_xepgia[['Kho_ID']])

C:\Users\phung\AppData\Local\Temp\ipykernel_22068\2307936470.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nienkhoa = pd.read_sql(query_Nienkyhoa, conn_dwh_library)


4        166
5        235
6        235
7        235
8        235
        ... 
71416     43
71417      1
71419     37
71420      1
71421     46
Name: Khoa_hoc, Length: 69310, dtype: int64


## Xử lý Gia_tien

In [ ]:
# Gia với Gia_tien giống nhau
# Gia kiểu varchar và Gia_tien kiểu money
# quét qua từng hàng
for index, row in df_xepgia.iterrows(): 
    if pd.isnull(row['Gia_tien']): # nếu gia tiền null và gia đang có giá trị thì chuyển về float và dán lại
        if not pd.isnull(row['Gia']):
            df_xepgia.at[index, 'Gia_tien'] = float(row['Gia'])
        else:
            df_xepgia.at[index, 'Gia_tien'] = 0;
print(df_xepgia['Gia_tien'])

C:\Users\phung\AppData\Local\Temp\ipykernel_22068\2376671368.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)


    Id  Dan_toc Mapping Mã  CSV Mã CSV Tên  \
0    1     Kinh        1.0     1.0    Kinh   
1    2    Mường        3.0     2.0     Tày   
2    3      Tày        2.0     3.0    Thái   
3    4     Thái        3.0     4.0     Hoa   
4    5      Hoa        4.0     5.0  Khơ-me   
..  ..      ...        ...     ...     ...   
68  71     Ê Đê       12.0     NaN     NaN   
69  72      Thổ        2.0     NaN     NaN   
70  73    Kờ Ho         56     NaN     NaN   
71  74     Jrai         56     NaN     NaN   
72  75  Châu mạ       28.0     NaN     NaN   

                                         CSV Tên khác  
0                                                việt  
1           thổ, ngạn, phén, thù lao, pa dí, tày khao  
2   tày đăm, tày mười, tày thanh, mán thanh, hàng ...  
3   hán, triều châu, phúc kiến, quảng đông, hải na...  
4              cur, cul, cu, thổ, việt gốc miên, krôm  
..                                                ...  
68                                                NaN  

## Load data

### [Nếu cần] Clear bảng

In [ ]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM DIM_Xep_gia"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [ ]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_dwh_library.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO DIM_Xep_gia (
                    ID_xep_gia, ID_tai_lieu, Ma_xep_gia,
                    ID_thu_vien, ID_kho, 
                    Ngay_bo_sung,
                    Cho_nhap_kho, InUsed,
                    Gia_tien, InCirculation, Kiem_ke,
                    Nguon_Nhap, Callnumber, So_HD
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """
# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['ID'], row['Tai_lieu_ID'], row['Ma_xep_gia'], 
        row['Ten_thu_vien_ID'], row['Kho_ID'], 
        row['Ngay_bo_sung'],
        row['Cho_nhap_kho'], row['InUsed'],
        row['Gia_tien'], row['InCirculation'], row['Kiem_ke'], 
        row['Nguon_Nhap'], row['Callnumber'], row['So_HD']
    )
    for index, row in df_xepgia.iterrows()
]
# Sử dụng executemany để chèn dữ liệu cùng lúc
cursor_dwh.executemany(insert_query, data_to_insert)
# Commit thay đổi
conn_dwh_library.commit()
# Đóng cursor và kết nối
cursor_dwh.close()
conn_dwh_library.close()
